In [1]:
import pandas as pd
from glob import glob
import re
from urlextract import URLExtract
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42  # For consistent language detection results
from transformers import pipeline, set_seed
set_seed(42) # Set seed for reproducibility
import os
os.makedirs('Outputs', exist_ok=True)

In [2]:
############ Filtering English Notes ############
df = pd.read_csv("../Data/Original CN/Notes/notes-00000.tsv", sep="\t", low_memory=False)#.sample(10000, random_state=42)

lang = []
for i in df.summary:
    try:
        lang.append(detect(i))
    except:
        lang.append(None)

df["summaryLang"] = lang
df = df[df.summaryLang=='en']
print(f"Number of English notes: {len(df)}")
df.to_csv("Outputs/en_notes.tsv", sep='\t', index=False)

Number of English notes: 1312173


In [3]:
############ MNLI based filtering ############
df = pd.read_csv("Outputs/en_notes.tsv", sep='\t', low_memory=False)

extractor = URLExtract()
def remove_links(text):
    urls = extractor.find_urls(text)
    for u in urls:
        text = text.replace(u, " ")
    text = re.sub(r'\s+', ' ', text).strip()

    if(text==''):
        return " "
    return text

CN = [remove_links(str(item)) for item in df.summary]

bs = 16
classifier = pipeline('zero-shot-classification', model="FacebookAI/roberta-large-mnli", device='cuda', batch_size=bs)
resp = []
# candidate_labels = ['United States Politics', 'Other']
for i in range(0, len(CN), bs):
    sequence_to_classify = list(CN[i:i+bs])
    outputs = classifier(sequence_to_classify, ['United States Politics', 'Other'])
    resp += outputs

for i in range(len(resp)):
    resp[i] = resp[i]['labels'][0]

df["isUSPolitics"] = resp
print("Classification stats (MNLI):-")
print(df.isUSPolitics.value_counts())

df.to_csv("Outputs/notes_MNLI.tsv", sep="\t", index=False)

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at FacebookAI/roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Classification stats (MNLI):-
isUSPolitics
Other                     1231473
United States Politics      80700
Name: count, dtype: int64


In [4]:
############ Keyword based filtering ############
df = pd.read_csv("Outputs/en_notes.tsv", sep='\t', low_memory=False)

us_politics_keywords = [
    "Donald Trump", "Trump", 
    "Joe Biden", "Biden", 
    "Barack Obama", "Obama", "Hillary Clinton", "Bernie Sanders",
    "Nancy Pelosi", "Ted Cruz", "Kamala Harris", "Kamala", 
    "Mitch McConnell", "Marco Rubio",
    "Rand Paul", "Bill Clinton", "Robert F. Kennedy Jr.", "Mike Pence", "Chuck Schumer",
    "Rudy Giuliani", "Chris Christie", "Donald Trump Jr.", "Mike Huckabee",
    "Elizabeth Warren", "Alexandria Ocasio-Cortez", "Tom Cotton", "Tucker Carlson",
    "Ron DeSantis", "Rick Scott", "Anthony Fauci", "Ben Carson", "Lindsey Graham",
    "Adam Schiff", "Kevin McCarthy", "Pete Buttigieg", "Nikki Haley", "JD Vance",
    "Marjorie Taylor Greene", "Ilhan Omar", "Robert Mueller", "Jim Jordan",
    "Michelle Obama", "Hunter Biden", "Ron Johnson", "Dick Durbin", "Mike Bloomberg",
    "James Comey", "Kirsten Gillibrand", "Cory Booker", "Mitt Romney", "John McCain", 
    "John Kerry", "Sarah Palin", "Rick Santorum",
    "Rick Perry", "Newt Gingrich", "Harry Reid", "Paul Ryan", "John Boehner",
    "Michele Bachmann", "Jeb Bush", "Jeff Sessions", "David Perdue", "Mary Landrieu",
    "Howard Dean", "Amy Klobuchar", "John Kasich", "Bill Cassidy", "Mark Pryor",
    "John Edwards", "Ron Paul", "Brett Kavanaugh", "David Axelrod", "Cory Gardner",
    "Martin O'Malley", "Scott Walker", "Scott Brown", "Charlie Crist",
    "Joni Ernst", "Gary Peters", "Kay Hagan", "Eric Cantor", "Pat Toomey",
    "Sharron Angle", "Lamar Smith"]
    ## There were more topic related keywords but this could lower the precision
    # "Affordable Care Act", "Benghazi", "Bush tax cuts", "Capitol riot",
    # "campaign finance", "campaign ads", "Clinton emails", "defense spending",
    # "gun control", "illegal immigration",
    # "impeachment", "impeachment inquiry", "IRS", "mail-in voting", "medicaid",
    # "medicare", "preexisting conditions", "Russia investigation", "social security",
    # "supreme court", "Tax Cuts and Jobs Act", "tax cuts", "taxes", "trade deficit",
    # "unemployment rate", "veterans", "voting", "border wall", "border security"


us_politics_keywords = [keyword.lower() for keyword in us_politics_keywords]

def contains_us_politics_keywords(text):
    text = text.lower()
    for keyword in us_politics_keywords:
        if keyword in text:
            return 'United States Politics'
    return 'Other'

CN = [str(item) for item in df.summary]
resp = []
for i in CN:
    resp.append(contains_us_politics_keywords(i))

df["isUSPolitics"] = resp
print("Classification stats (Keywords):-")
print(df.isUSPolitics.value_counts())

df.to_csv("Outputs/notes_Keywords.tsv", sep="\t", index=False)

Classification stats (Keywords):-
isUSPolitics
Other                     1209844
United States Politics     102329
Name: count, dtype: int64


In [5]:
############ Combined (MNLI + Keywords) ############
df_mnli = pd.read_csv("Outputs/notes_MNLI.tsv", delimiter="\t", low_memory=False)
df_keyw = pd.read_csv("Outputs/notes_Keywords.tsv", delimiter="\t", low_memory=False)

resp = []
for m, k in zip(df_mnli.isUSPolitics, df_keyw.isUSPolitics):
    if m == 'United States Politics':
        resp.append(m)
    elif k == 'United States Politics':
        resp.append(k)
    else:
        resp.append('Other')

df_mnli["isUSPolitics"] = resp
print("Classification stats (MNLI+Keywords):-")
print(df_mnli.isUSPolitics.value_counts())

df_mnli.to_csv("Outputs/notes_MNLI_Keywords.tsv", sep="\t", index=False)

Classification stats (MNLI+Keywords):-
isUSPolitics
Other                     1173079
United States Politics     139094
Name: count, dtype: int64


In [6]:
# Based on notes texts, we filter out the tweets that are not related to US Politics
df_filtered = pd.read_csv("Outputs/notes_MNLI_Keywords.tsv", delimiter="\t", low_memory=False)

df_filtered = df_filtered[df_filtered.isUSPolitics == "United States Politics"]

filtered_tweetIds = df_filtered.tweetId.unique()

print(len(filtered_tweetIds)) # 77650

101198


In [7]:
# Removing tweets that have media notes (out of scope for text-only benchmarking)
df_notes = pd.read_csv("../Data/Original CN/Notes/notes-00000.tsv", delimiter="\t", low_memory=False)
df_notes_filtered = df_notes[df_notes.tweetId.isin(filtered_tweetIds)]
df_notes_filtered = df_notes_filtered.groupby('tweetId')['isMediaNote'].mean()
filtered_tweetIds_notMediaNotes = df_notes_filtered[df_notes_filtered==0].index

print(len(filtered_tweetIds_notMediaNotes)) # 72919

93455


In [8]:
# Loading filtered notes and rating files
df_notes = df_notes[df_notes.tweetId.isin(filtered_tweetIds_notMediaNotes)]

rating_files = glob("../Data/Original CN/Ratings/*")

df_r = None
for f in rating_files:
    if df_r is None:
        # first intialization
        df_r = pd.read_csv(f, sep="\t")
    else:
        tmp = pd.read_csv(f, sep="\t")
        df_r = pd.concat([df_r, tmp], ignore_index=True)

df_r = df_r[df_r.noteId.isin(df_notes.noteId)]
df_r.columns

Index(['noteId', 'raterParticipantId', 'createdAtMillis', 'version', 'agree',
       'disagree', 'helpful', 'notHelpful', 'helpfulnessLevel', 'helpfulOther',
       'helpfulInformative', 'helpfulClear', 'helpfulEmpathetic',
       'helpfulGoodSources', 'helpfulUniqueContext', 'helpfulAddressesClaim',
       'helpfulImportantContext', 'helpfulUnbiasedLanguage', 'notHelpfulOther',
       'notHelpfulIncorrect', 'notHelpfulSourcesMissingOrUnreliable',
       'notHelpfulOpinionSpeculationOrBias', 'notHelpfulMissingKeyPoints',
       'notHelpfulOutdated', 'notHelpfulHardToUnderstand',
       'notHelpfulArgumentativeOrBiased', 'notHelpfulOffTopic',
       'notHelpfulSpamHarassmentOrAbuse', 'notHelpfulIrrelevantSources',
       'notHelpfulOpinionSpeculation', 'notHelpfulNoteNotNeeded',
       'ratedOnTweetId'],
      dtype='object')

In [9]:
# Concatenating ratings for each noteId: total, helpful, somewhat helpful, not helpful
def return_rating(noteId):
    tmp_r = df_r[df_r.noteId == noteId]

    return [len(tmp_r), 
            len(tmp_r[tmp_r.helpful == 1]) + len(tmp_r[tmp_r.helpfulnessLevel == "HELPFUL"]), 
            len(tmp_r[tmp_r.helpfulnessLevel == "SOMEWHAT_HELPFUL"]), 
            len(tmp_r[tmp_r.notHelpful == 1]) + len(tmp_r[tmp_r.helpfulnessLevel == "NOT_HELPFUL"])]

totalRatings, helpfulRatings, somewhatHelpfulRatings, notHelpfulRatings = [], [], [], []
for noteId in df_notes.noteId:
    ratings = return_rating(noteId)
    totalRatings.append(ratings[0])
    helpfulRatings.append(ratings[1])
    somewhatHelpfulRatings.append(ratings[2])
    notHelpfulRatings.append(ratings[3])
    
df_notes["totalRatings"] = totalRatings
df_notes["helpfulRatings"] = helpfulRatings
df_notes["somewhatHelpfulRatings"] = somewhatHelpfulRatings
df_notes["notHelpfulRatings"] = notHelpfulRatings

In [10]:
# Adding current helpfulness status of each note
df_status = pd.read_csv("../Data/Original CN/StatusHistory/noteStatusHistory-00000.tsv", sep="\t", low_memory=False)
df_status.index = df_status.noteId

status = []
for i in df_notes.noteId:
    try:
        tmp = df_status.loc[i]
        status.append(tmp.currentStatus)
    except KeyError:
        status.append("NA")

df_notes["helpfulnessStatus"] = status

print("Stats of current helpfulness status of notes:")
print(df_notes.helpfulnessStatus.value_counts())

# Saving the filtered notes with ratings
df_notes.to_csv("Outputs/notes.tsv", sep="\t", index=False)

Stats of current helpfulness status of notes:
helpfulnessStatus
NEEDS_MORE_RATINGS             176226
CURRENTLY_RATED_HELPFUL          8425
CURRENTLY_RATED_NOT_HELPFUL      7979
NA                               4266
Name: count, dtype: int64


In [11]:
# Test: after running the entire notebook once, rerun it again
# This will store the final notes DataFrame in a variable
# When you will run it should print True (first and seccond run should be the same)
# Pro tip: you can check this for a sample -- uncomment the sample option in cell 2.

try:
    # Check if the final notes DataFrame is the same as the one in previous run
    print(df_notes.equals(old_df_notes))
except NameError:
    old_df_notes = df_notes.copy()
    print("Rerun the notebook without Restart kernel")

Rerun the notebook without Restart kernel
